# Yazılım Hata Riski ve Test Önceliklendirme

## Amaç
Yazılım modüllerine ait ölçümleri kullanarak hata içeren modülleri
tahmin etmek ve sınırlı inceleme bütçesinde test önceliklendirmesine
katkısını değerlendirmek.

## Veri kaynağı
https://www.kaggle.com/datasets/semustafacevik/software-defect-prediction

## Çalışma ilkeleri
- Orijinal veri değiştirilmeden saklanacaktır.
- Ön işleme kararları gerekçeleriyle açıklanacaktır.
- Model ve karar eşiği seçiminde test kümesi kullanılmayacaktır.
- Sonuçlar, sınırlılıklar ve başarısız denemeler dürüstçe raporlanacaktır.


In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path

PROJE_KLASORU = Path(
    "/content/drive/MyDrive/Yazilim_Hata_Riski"
)

VERI_KLASORU = PROJE_KLASORU / "data"
CIKTI_KLASORU = PROJE_KLASORU / "outputs"
RAPOR_KLASORU = PROJE_KLASORU / "reports"

for klasor in [VERI_KLASORU, CIKTI_KLASORU, RAPOR_KLASORU]:
    klasor.mkdir(parents=True, exist_ok=True)

csv_dosyalari = sorted(VERI_KLASORU.glob("*.csv"))

print("Proje klasörü:", PROJE_KLASORU)
print("\nBulunan CSV dosyaları:")

for dosya in csv_dosyalari:
    print("-", dosya.name)

if not csv_dosyalari:
    raise FileNotFoundError(
        "data klasöründe CSV bulunamadı. "
        "ZIP içinden çıkardığın CSV dosyasını bu klasöre yükle."
    )

Proje klasörü: /content/drive/MyDrive/Yazilim_Hata_Riski

Bulunan CSV dosyaları:
- jm1.csv


In [3]:
import hashlib
import json
from datetime import datetime, timezone

import pandas as pd

VERI_DOSYASI = VERI_KLASORU / "jm1.csv"

if not VERI_DOSYASI.exists():
    raise FileNotFoundError(
        f"Dosya bulunamadı: {VERI_DOSYASI.name}. "
        "Önceki hücrede listelenen dosya adını kontrol et."
    )

# Aynı veri dosyasını kullandığımızı doğrulamak için parmak izi.
sha256 = hashlib.sha256()

with VERI_DOSYASI.open("rb") as dosya:
    for parca in iter(lambda: dosya.read(1024 * 1024), b""):
        sha256.update(parca)

# Orijinal veriyi ayrı tutuyoruz.
df_ham = pd.read_csv(VERI_DOSYASI)
df = df_ham.copy()

veri_kaydi = {
    "kaynak": (
        "https://www.kaggle.com/datasets/"
        "semustafacevik/software-defect-prediction"
    ),
    "dosya_adi": VERI_DOSYASI.name,
    "sha256": sha256.hexdigest(),
    "satir_sayisi": int(df_ham.shape[0]),
    "sutun_sayisi": int(df_ham.shape[1]),
    "okuma_zamani_utc": datetime.now(timezone.utc).isoformat(),
}

with (CIKTI_KLASORU / "veri_kaydi.json").open(
    "w", encoding="utf-8"
) as dosya:
    json.dump(veri_kaydi, dosya, ensure_ascii=False, indent=2)

print("Veri başarıyla okundu.")
print("Satır ve sütun sayısı:", df.shape)
print("SHA-256:", veri_kaydi["sha256"])

display(df.head())

Veri başarıyla okundu.
Satır ve sütun sayısı: (10885, 22)
SHA-256: 7f8aff7353f7932c0d058011b554066375e52ef4d8a385454f957b0ad2b60094


,loc,v(g),ev(g),iv(g),n,v,l,d,i,e,...,lOCode,lOComment,lOBlank,locCodeAndComment,uniq_Op,uniq_Opnd,total_Op,total_Opnd,branchCount,defects
0,1.1,1.4,1.4,1.4,1.3,1.30,1.30,1.30,1.30,1.30,...,2,2,2,2,1.2,1.2,1.2,1.2,1.4,False
1,1.0,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.00,1.00,...,1,1,1,1,1,1,1,1,1,True
2,72.0,7.0,1.0,6.0,198.0,1134.13,0.05,20.31,55.85,23029.10,...,51,10,8,1,17,36,112,86,13,True
3,190.0,3.0,1.0,3.0,600.0,4348.76,0.06,17.06,254.87,74202.67,...,129,29,28,2,17,135,329,271,5,True
4,37.0,4.0,1.0,4.0,126.0,599.12,0.06,17.19,34.86,10297.30,...,28,1,6,0,11,16,76,50,7,True


In [4]:
print("SÜTUNLAR")
print(df.columns.tolist())

print("\nVERİ YAPISI")
df.info()

print("\nPANDAS TARAFINDAN ALGILANAN EKSİK DEĞERLER")
display(df.isna().sum().sort_values(ascending=False))

print("\nTAMAMEN AYNI OLAN TEKRAR SATIR SAYISI")
print(df.duplicated().sum())

print("\nSAYISAL DEĞİŞKENLERİN ÖZETİ")
display(df.describe().T)

SÜTUNLAR
['loc', 'v(g)', 'ev(g)', 'iv(g)', 'n', 'v', 'l', 'd', 'i', 'e', 'b', 't', 'lOCode', 'lOComment', 'lOBlank', 'locCodeAndComment', 'uniq_Op', 'uniq_Opnd', 'total_Op', 'total_Opnd', 'branchCount', 'defects']

VERİ YAPISI
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10885 entries, 0 to 10884
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   loc                10885 non-null  float64
 1   v(g)               10885 non-null  float64
 2   ev(g)              10885 non-null  float64
 3   iv(g)              10885 non-null  float64
 4   n                  10885 non-null  float64
 5   v                  10885 non-null  float64
 6   l                  10885 non-null  float64
 7   d                  10885 non-null  float64
 8   i                  10885 non-null  float64
 9   e                  10885 non-null  float64
 10  b                  10885 non-null  float64
 11  t                  10885 non-null  

,0
loc,0
v(g),0
ev(g),0
iv(g),0
n,0
v,0
l,0
d,0
i,0
e,0



TAMAMEN AYNI OLAN TEKRAR SATIR SAYISI
1973

SAYISAL DEĞİŞKENLERİN ÖZETİ


,count,mean,std,min,25%,50%,75%,max
loc,10885.0,42.016178,76.593332,1.0,11.00,23.00,46.00,3442.00
v(g),10885.0,6.348590,13.019695,1.0,2.00,3.00,7.00,470.00
ev(g),10885.0,3.401047,6.771869,1.0,1.00,1.00,3.00,165.00
iv(g),10885.0,4.001599,9.116889,1.0,1.00,2.00,4.00,402.00
n,10885.0,114.389738,249.502091,0.0,14.00,49.00,119.00,8441.00
v,10885.0,673.758017,1938.856196,0.0,48.43,217.13,621.48,80843.08
l,10885.0,0.135335,0.160538,0.0,0.03,0.08,0.16,1.30
d,10885.0,14.177237,18.709900,0.0,3.00,9.09,18.90,418.20
i,10885.0,29.439544,34.418313,0.0,11.86,21.93,36.78,569.78
e,10885.0,36836.365343,434367.801255,0.0,161.94,2031.02,11416.43,31079782.27


## 2. Veri Kalitesi Denetimi

İlk incelemede beş girdi sütununun metin türünde okunduğu ve
1.973 tekrar satır bulunduğu görülmüştür.

Bu aşamada:
- Sayıya dönüştürülemeyen değerler belirlenecek.
- Hedef sınıfların dağılımı incelenecek.
- Aynı girdilere sahip kayıtların etiketleri karşılaştırılacak.
- Şüpheli sayım değerleri raporlanacaktır.

Henüz satır silinmeyecek ve eksik değer doldurulmayacaktır.

In [5]:
# Her çalıştırmada orijinal veriden başlayarak aynı sonucu üretelim.
df_kontrol = df_ham.copy()

HEDEF = "defects"
ozellikler = [sutun for sutun in df_kontrol.columns if sutun != HEDEF]

donusum_raporu = []

for sutun in ozellikler:
    orijinal = df_kontrol[sutun]
    sayisal = pd.to_numeric(orijinal, errors="coerce")

    # Orijinalde dolu olduğu hâlde sayıya çevrilemeyen değerler.
    sorunlu = orijinal.notna() & sayisal.isna()

    donusum_raporu.append({
        "sutun": sutun,
        "ilk_veri_turu": str(orijinal.dtype),
        "sayiya_cevrilemeyen_adet": int(sorunlu.sum()),
        "sorunlu_degerler": str(
            orijinal.loc[sorunlu].astype(str).unique().tolist()
        ),
        "toplam_eksik": int(sayisal.isna().sum()),
    })

    df_kontrol[sutun] = sayisal

donusum_raporu = pd.DataFrame(donusum_raporu)

display(donusum_raporu)

donusum_raporu.to_csv(
    CIKTI_KLASORU / "sayisal_donusum_denetimi.csv",
    index=False
)

,sutun,ilk_veri_turu,sayiya_cevrilemeyen_adet,sorunlu_degerler,toplam_eksik
0,loc,float64,0,[],0
1,v(g),float64,0,[],0
2,ev(g),float64,0,[],0
3,iv(g),float64,0,[],0
4,n,float64,0,[],0
5,v,float64,0,[],0
6,l,float64,0,[],0
7,d,float64,0,[],0
8,i,float64,0,[],0
9,e,float64,0,[],0


In [6]:
print("Hedef sütunun veri türü:", df_kontrol[HEDEF].dtype)
print("Hedef sütundaki eksik değer:", df_kontrol[HEDEF].isna().sum())

sinif_tablosu = (
    df_kontrol[HEDEF]
    .value_counts(dropna=False)
    .rename_axis("defects")
    .reset_index(name="adet")
)

sinif_tablosu["yuzde"] = (
    100 * sinif_tablosu["adet"] / len(df_kontrol)
).round(2)

display(sinif_tablosu)

sinif_tablosu.to_csv(
    CIKTI_KLASORU / "ham_veri_sinif_dagilimi.csv",
    index=False
)

Hedef sütunun veri türü: bool
Hedef sütundaki eksik değer: 0


,defects,adet,yuzde
0,False,8779,80.65
1,True,2106,19.35


In [7]:
# Bütün sütunları aynı olan kayıtlar.
tam_tekrar = df_kontrol.duplicated()

# Hedef hariç bütün girdileri aynı olan kayıtlar.
ayni_girdi = df_kontrol.duplicated(
    subset=ozellikler,
    keep=False
)

# Aynı girdiler için hem True hem False etiketi var mı?
etiket_cesidi = (
    df_kontrol
    .groupby(ozellikler, dropna=False)[HEDEF]
    .transform("nunique")
)

celiskili = etiket_cesidi.gt(1)

tekrar_ozeti = pd.DataFrame({
    "kontrol": [
        "İlk kopya dışındaki tam tekrar satırları",
        "Aynı girdiye sahip gruplardaki tüm satırlar",
        "Çelişkili etiket grubundaki tüm satırlar",
    ],
    "adet": [
        int(tam_tekrar.sum()),
        int(ayni_girdi.sum()),
        int(celiskili.sum()),
    ],
})

display(tekrar_ozeti)

if celiskili.any():
    print("Çelişkili etiketlere ait örnek kayıtlar:")
    display(df_kontrol.loc[celiskili].head(10))

tekrar_ozeti.to_csv(
    CIKTI_KLASORU / "tekrar_ve_etiket_denetimi.csv",
    index=False
)

,kontrol,adet
0,İlk kopya dışındaki tam tekrar satırları,1973
1,Aynı girdiye sahip gruplardaki tüm satırlar,2730
2,Çelişkili etiket grubundaki tüm satırlar,889


Çelişkili etiketlere ait örnek kayıtlar:


,loc,v(g),ev(g),iv(g),n,v,l,d,i,e,...,lOCode,lOComment,lOBlank,locCodeAndComment,uniq_Op,uniq_Opnd,total_Op,total_Opnd,branchCount,defects
8,24.0,2.0,1.0,2.0,0.0,0.00,0.00,0.00,0.00,0.00,...,0,0,0,0,0.0,0.0,0.0,0.0,3.0,True
60,8.0,1.0,1.0,1.0,14.0,46.51,0.33,3.00,15.50,139.52,...,4,0,2,0,5.0,5.0,8.0,6.0,1.0,True
74,20.0,1.0,1.0,1.0,0.0,0.00,0.00,0.00,0.00,0.00,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,True
78,12.0,3.0,1.0,2.0,22.0,88.00,0.12,8.33,10.56,733.33,...,9,0,1,0,10.0,6.0,12.0,10.0,5.0,True
86,18.0,1.0,1.0,1.0,18.0,72.00,0.30,3.30,21.82,237.60,...,14,0,2,0,6.0,10.0,7.0,11.0,1.0,True
90,4.0,1.0,1.0,1.0,10.0,30.00,0.50,2.00,15.00,60.00,...,2,0,0,0,4.0,4.0,6.0,4.0,1.0,True
91,4.0,1.0,1.0,1.0,10.0,30.00,0.50,2.00,15.00,60.00,...,2,0,0,0,4.0,4.0,6.0,4.0,1.0,True
92,4.0,1.0,1.0,1.0,10.0,30.00,0.50,2.00,15.00,60.00,...,2,0,0,0,4.0,4.0,6.0,4.0,1.0,True
93,4.0,1.0,1.0,1.0,10.0,30.00,0.50,2.00,15.00,60.00,...,2,0,0,0,4.0,4.0,6.0,4.0,1.0,True
94,4.0,1.0,1.0,1.0,10.0,30.00,0.50,2.00,15.00,60.00,...,2,0,0,0,4.0,4.0,6.0,4.0,1.0,True


In [8]:
import numpy as np

# Adları ve ilk inceleme doğrultusunda tam sayı olması beklenen
# ölçümleri işaretliyoruz. Bu kontrol tek başına silme nedeni değildir.
sayim_sutunlari = [
    "loc", "v(g)", "ev(g)", "iv(g)", "n",
    "lOCode", "lOComment", "lOBlank", "locCodeAndComment",
    "uniq_Op", "uniq_Opnd", "total_Op", "total_Opnd",
    "branchCount"
]

ondalik_maskesi = pd.DataFrame(
    False,
    index=df_kontrol.index,
    columns=sayim_sutunlari
)

for sutun in sayim_sutunlari:
    degerler = df_kontrol[sutun]

    ondalik_maskesi[sutun] = (
        degerler.notna()
        & ~np.isclose(degerler, np.round(degerler))
    )

supheli_satir = ondalik_maskesi.any(axis=1)

print(
    "En az bir sayım ölçümü ondalıklı olan satır sayısı:",
    int(supheli_satir.sum())
)

display(
    ondalik_maskesi.sum()
    .rename("ondalik_deger_adedi")
    .to_frame()
)

display(df_kontrol.loc[supheli_satir].head(10))

df_kontrol.loc[supheli_satir].to_csv(
    CIKTI_KLASORU / "incelenecek_ondalik_sayim_kayitlari.csv",
    index_label="orijinal_satir_indeksi"
)

En az bir sayım ölçümü ondalıklı olan satır sayısı: 1


,ondalik_deger_adedi
loc,1
v(g),1
ev(g),1
iv(g),1
n,1
lOCode,0
lOComment,0
lOBlank,0
locCodeAndComment,0
uniq_Op,1


,loc,v(g),ev(g),iv(g),n,v,l,d,i,e,...,lOCode,lOComment,lOBlank,locCodeAndComment,uniq_Op,uniq_Opnd,total_Op,total_Opnd,branchCount,defects
0,1.1,1.4,1.4,1.4,1.3,1.3,1.3,1.3,1.3,1.3,...,2,2,2,2,1.2,1.2,1.2,1.2,1.4,False


In [9]:
# Önceki denetim hücrelerinin çalıştığını kontrol et.
gerekli_degiskenler = [
    "df_kontrol",
    "donusum_raporu",
    "sinif_tablosu",
    "tekrar_ozeti",
    "celiskili",
    "supheli_satir",
]

eksik_degiskenler = [
    isim for isim in gerekli_degiskenler
    if isim not in globals()
]

if eksik_degiskenler:
    raise RuntimeError(
        "Önce veri denetimi hücrelerini sırayla çalıştır. "
        f"Eksik değişkenler: {eksik_degiskenler}"
    )

print("1. SAYIYA DÖNÜŞTÜRME SORUNLARI")
display(
    donusum_raporu.loc[
        donusum_raporu["toplam_eksik"].gt(0)
    ]
)

print("\n2. HEDEF SINIF DAĞILIMI")
display(sinif_tablosu)

print("\n3. TEKRAR VE ETİKET DENETİMİ")
display(tekrar_ozeti)

print("\n4. ŞÜPHELİ SAYIM KAYITLARI")
print("Satır sayısı:", int(supheli_satir.sum()))
display(df_kontrol.loc[supheli_satir].head(10))

# Veri üzerinde yaptığımız dönüşümü, eksikleri doldurmadan kaydet.
# Orijinal CSV dosyasının üzerine yazılmaz.
df_kontrol.to_csv(
    CIKTI_KLASORU / "sayisala_donusturulmus_veri.csv",
    index=False
)

rapor_notu = f"""# Veri Kalitesi Denetimi — Ara Bulgular

- Ham veri: {df_ham.shape[0]} satır, {df_ham.shape[1]} sütun.
- Hedef değişken: defects.
- Girdi sayısı: {len(ozellikler)}.
- Ham veride ilk kopya dışındaki tam tekrar sayısı:
  {int(df_ham.duplicated().sum())}.
- Sayısala dönüşümden sonra eksik değer içeren satır sayısı:
  {int(df_kontrol[ozellikler].isna().any(axis=1).sum())}.
- Çelişkili etiket gruplarındaki toplam satır sayısı:
  {int(celiskili.sum())}.
- En az bir sayım ölçümü ondalıklı olan satır sayısı:
  {int(supheli_satir.sum())}.

## Bu aşamada uygulanan işlemler
- Orijinal veri korunmuştur.
- Girdi sütunları sayısala dönüştürülmüştür.
- Sayıya dönüştürülemeyen değerler eksik olarak işaretlenmiştir.
- Henüz kayıt silinmemiş, eksik değer doldurulmamış
  ve model eğitilmemiştir.

## Sonraki kararlar
- Tekrar ve çelişkili etiket politikası belirlenecek.
- Şüpheli kayıtlar değerlendirilerek gerekçeli karar verilecek.
- Aynı girdilerin farklı veri kümelerine geçmesi engellenecek.
- Eksik değer doldurma ve ölçekleme eğitim verisinde öğrenilecek.

Bu dosya ara denetim kaydıdır; nihai sonuç raporu değildir.
"""

not_dosyasi = RAPOR_KLASORU / "veri_denetimi_ara_notlari.md"
not_dosyasi.write_text(rapor_notu, encoding="utf-8")

print("\nDenetim verisi ve rapor notları Drive'a kaydedildi.")

1. SAYIYA DÖNÜŞTÜRME SORUNLARI


,sutun,ilk_veri_turu,sayiya_cevrilemeyen_adet,sorunlu_degerler,toplam_eksik
16,uniq_Op,object,5,['?'],5
17,uniq_Opnd,object,5,['?'],5
18,total_Op,object,5,['?'],5
19,total_Opnd,object,5,['?'],5
20,branchCount,object,5,['?'],5



2. HEDEF SINIF DAĞILIMI


,defects,adet,yuzde
0,False,8779,80.65
1,True,2106,19.35



3. TEKRAR VE ETİKET DENETİMİ


,kontrol,adet
0,İlk kopya dışındaki tam tekrar satırları,1973
1,Aynı girdiye sahip gruplardaki tüm satırlar,2730
2,Çelişkili etiket grubundaki tüm satırlar,889



4. ŞÜPHELİ SAYIM KAYITLARI
Satır sayısı: 1


,loc,v(g),ev(g),iv(g),n,v,l,d,i,e,...,lOCode,lOComment,lOBlank,locCodeAndComment,uniq_Op,uniq_Opnd,total_Op,total_Opnd,branchCount,defects
0,1.1,1.4,1.4,1.4,1.3,1.3,1.3,1.3,1.3,1.3,...,2,2,2,2,1.2,1.2,1.2,1.2,1.4,False



Denetim verisi ve rapor notları Drive'a kaydedildi.
